# SQLAlchemy: jointures, requêtes complexes et opérations CRUD

On a vu dans le cours précédent:
- comment faire des requêtes SQL (textuelles et ORM)
- comment décrire des tables individuelles via des `db.Models` SQLAlchemy
- comment faire des requêtes `SELECT` sur une table

Aujourd'hui, on va voir: 
- **les jointures SQL** et tables de relation
- **comment utiliser les jointures en SQLAlchemy**
- le reste des requêtes CRUD: **read**, **update** et **delete**


In [2]:
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)
print(db)

/home/paul/Documents/cours/tnah_devapp/richelieu.db
<SQLAlchemy>



--- 

# Modéliser des jointures

Si on reprend le dernier cours, on a modélisé deux tables: `Iconography` et `Author`.

```py
# comment lit-on chacun des attributs ci-dessous ?
class Iconography(db.Model):
    __tablename__ = "Iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    # NOTE: il reste à définir la relation avec la table `author`, mais on verra comment faire plus tard !
    id_author = ...


class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)
```

Si on reprend notre modèle de base de données ci-dessous, on voit plus globalement qu'on a deux types de relations:
- **one to many**: `Author <-> Iconography`
- **many to many**: `Iconography <-> Place` et `Iconography <-> Theme`
- (et pour rappel, les relations *many to one*, c'est la même chose que des relations *one to many* mais à l'envers)

![db schema](./img/db_schema.png)

## One to many: `Author <-> Iconography`

`author` a une relation one to many à `iconography`:
- 1 ressource iconographique a un.e seul.e auteur.ice, 
- mais une entrée de la table `author` peut être associée à plusieurs ressources iconographiques.

**Pour rappel, en SQL, cela est modélisé par une clé étrangère sur la table `iconography`** qui pointe vers `author`: `iconography.id_author`.

**Avec SQLAlchemy**, pour décrire une relation one-to-many, on doit définir:
- **la colonne `Iconography.id_author`**, une `ForeignKey` vers `author.id`.
- **la propriété `Iconography.author`**, qui permettra d'accéder à l'auteur.ice d'une ressource icono
- **la propriété `Author.iconography`**, qui permettra d'accéder aux ressources iconographiques associées à un.e auteur.ice.

```py
class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author"
    )

class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[int] = mapped_column(ForeignKey("author.id"))
    
    author: Mapped["Author"] = relationship(
        back_populates="iconography"
    )
```

### Modéliser une clé étrangère: `Iconography.id_author`

Voici comment on définit une clé étrangère:

```py
class Iconography(db.Model):
    id_author: Mapped[int] = mapped_column(ForeignKey("author.id"))
```

On définit en général une clé étrangère comme n'importe quelle colonne:
- `Mapped[int]` indique le type de la colonne (entier non-nullable)
- `mapped_column()` définit les contraintes de la colonne 
- **la particularité est: `ForeignKey("author.id")`**: on définit le contenu de la colonne comme une clé étrangère qui pointe vers la colonne `author.id`.

### Modéliser les `relationships`: `Iconography.author` et `Author.iconography`

Contrairement à `iconography.id_author`, ces propriétés contiennent du nouveau:

```py
class Iconography(db.Model):
    author: Mapped["Author"] = relationship(
        back_populates="iconography"
    )

class Author(db.Model):
    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author"
    )
```

`Iconography.author` et `Author.iconography` **permettront d'accéder aux objets liés**: 
- `Iconography.author` contient l'objet `Author` associé à une ressource iconographique
- `Author.iconography` contient la liste d'objets `Iconography` associés à un `Author`
- **à noter**: `Iconography.author` et `Author.iconography` **ne sont pas des colonnes**: c'est des propriétés propres à SQLAlchemy et qui permettent de **ne pas avoir à faire de jointures à la main**

**Pour la syntaxe**, prenons:

```py
# dans la classe `Iconography`
author: Mapped["Author"] = relationship(
    back_populates="iconography"
)
```

- `Mapped["Author"]`: chaque objet `Iconography` est **associé à un seul objet `Author`**
- `relationship()`: 
    - on définit ce champ comme une **relation SQLAlchemy** (renvoi vers les objets d'un autre modèle)
    - `back_populates="iconography"`: indique que **le champ `Author.iconography` est lié à `Iconography.author`** (qu'on a aussi défini). Cela permet à SQLAlchemy de **synchroniser les valeurs de ces deux propriétés**.

À partir de là, comment interpréter `iconography` ci-dessous ?
```py
# dans la classe `Author`
iconography: Mapped[List["Iconography"]] = relationship(
    back_populates="author"
)
```

## Many-to-many: `Iconography <-> Place`

La manière de faire des relations many-to-many est très semblable, sauf que l'on doit aussi définir une *table secondaire*: la table de relation.

**On va modéliser la relation entre `Iconography` et `Place`**.

**En SQLAlchemy, pour modéliser une relation one-to-many**, il faut:
- définir `IconographyPlace`, la table de relation entre `Iconography` et `Place`
- définir `Iconography.place`, la propriété permettant d'accéder aux objets `Place` depuis `Iconography`
- définir `Place.iconography.`, la propriété permettant d'accéder aux objets `Iconography` depuis `Place` (inverse de `Iconography.place`, donc).

### La table secondaire: `IconographyPlace`

Voici notre modèle pour `IconographyPlace`, table de relation entre `Iconography` et `Place`. Rien de bien surprenant ici.

```py
class IconographyPlace(db.Model):
    __tablename__ = "iconography_place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_place: Mapped[int] = mapped_column(ForeignKey("place.id"))
```

### Définir la relation entre `Iconography` et `Place`: `Iconography.place` et `Place.iconography`

En SQLAlchemy, on définit une relation many to many **directement entre les tables `Iconography` et `Place`**. `IconographyPlace` est utilisé implicitement par SQLAlchemy.

```py
class Iconography(db.Model):
    __tablename__ = "iconography"
    # ...
    # NOTE: pas de delete-orphan sur Place: on veut qu'une Place continue d'exister même si aucune Iconography n'y fait référence
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )
```

- `place` est la propriété d'`Iconography` qui permet d'accéder aux objets `Place` 
- `Mapped[List["Place"]]`: `Iconography.place` est une liste d'objets `Place`
- `relationship()`: 
    - `back_populates="iconography"`: le champ `Iconography.place` est associé à `Place.iconography`.
    - `secondary=IconographyPlace.__table__`: `secondary` permet de définir une **une table de relation entre `Iconography` et `Place`**: `IconographyPlace`
        - cela veut dire que le lien `Iconography <-> IconographyPlace <-> Place` n'a pas besoin d'être défini: `IconographyPlace` est implicitement géré par SQLAlchemy.
        - on note que on utilise `IconographyPlace.__table__`, pas juste `IconographyPlace`

**`Place` est défini de la même manière**:

```py
class Place(db.Model):
    __tablename__ = "place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    address: Mapped[Optional[str]]
    richelieu_url: Mapped[str] = mapped_column(unique=True)
    # JSON est un type SQLAlchemy
    loc: Mapped[Dict] = mapped_column(JSON)
    plot: Mapped[Dict] = mapped_column(JSON)
    date_lower: Mapped[int]
    date_upper: Mapped[int]

    # comment interpétez vous cette propriété ?
    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="place"
    )
```

### Exercice: modéliser la relation many-to-many entre `Iconography` et `Theme`

À partir des tables `Iconography` et `Theme` définies ci-dessous:
- modélisez la table `IconographyTheme`
- complétez `Iconography` et `Theme` pour décrire la relation entre les deux modèles.

In [10]:
from typing import List, Dict, Optional

from sqlalchemy import JSON, ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship, Mapped


APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)


# à vous de jouer !
class IconographyTheme(db.Model):
    ...

# définir `Theme` et sa relation avec `Iconography` . Voir le modèle de `theme` dans le diagramme au début du cours. 
class Theme(db.Model):
    ...

# et complétez Iconography pour faire le lien avec `Theme`.
class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[int] = mapped_column(ForeignKey("author.id"))
    
    author: Mapped["Author"] = relationship(
        back_populates="iconography"
    )
    # NOTE: pas de delete-orphan sur Place: on veut qu'une Place continue d'exister même si aucune Iconography n'y fait référence
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )



ArgumentError: Mapper Mapper[IconographyTheme(iconography_theme)] could not assemble any primary key columns for mapped table 'iconography_theme'


## Many-to-one

---

# Requêter des jointures dans des requêtes `SELECT`

---

# CRUD

> **Note**: les opérations *create*, *update* et *delete* modifient notre base de données `richelieu.db`. Pour restaurer son état entre deux exercices, faire:
> ```bash
> cp richelieu.db.bak richelieu.db
> ```

In [ ]:
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)
print(db)